In [1]:
# Import necessary libraries
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set a style for plots
sns.set_style("whitegrid")

# Check for GPU availability and set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
# Function to get data loaders
def get_dataloaders(batch_size, num_workers, subset_size=10000):
    transform = transforms.Compose(
        [transforms.ToTensor(),
         transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

    # Download and load the training data
    full_trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                                download=True, transform=transform)
    # Create a subset
    trainset = Subset(full_trainset, range(subset_size))

    # Create data loaders
    trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True,
                             num_workers=num_workers, pin_memory=True if device.type == 'cuda' else False)

    return trainloader

# --- Model Definitions ---

# Small Model
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(16 * 16 * 16, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = x.view(-1, 16 * 16 * 16)
        x = self.fc1(x)
        return x

# Medium Model
class MediumCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Linear(64 * 16 * 16, 10)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

# Large Model
class LargeCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Linear(256 * 8 * 8, 10)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

# Helper function to get model by name
def get_model(name):
    if name == 'small':
        return SmallCNN()
    elif name == 'medium':
        return MediumCNN()
    elif name == 'large':
        return LargeCNN()
    else:
        raise ValueError("Unknown model name")

In [3]:
def run_experiment(model_name, target_device, batch_size, epochs, num_workers):
    """Runs a single training experiment and returns performance metrics."""
    # Setup
    model = get_model(model_name).to(target_device)
    trainloader = get_dataloaders(batch_size, num_workers)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

    # Training loop
    epoch_times = []
    total_start_time = time.time()

    # Reset CUDA memory stats
    if target_device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(target_device)

    for epoch in range(epochs):
        epoch_start_time = time.time()
        model.train()
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(target_device), labels.to(target_device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        epoch_end_time = time.time()
        epoch_times.append(epoch_end_time - epoch_start_time)

    total_training_time = time.time() - total_start_time
    avg_epoch_time = np.mean(epoch_times)

    # Get GPU memory usage
    gpu_mem_usage = 0
    if target_device.type == 'cuda':
        gpu_mem_usage = torch.cuda.max_memory_allocated(target_device) / (1024 ** 2) # in MB

    print(f"Finished Training: Model={model_name}, Device={target_device.type}, Batch={batch_size}, Workers={num_workers}")
    print(f"  Avg Epoch Time: {avg_epoch_time:.2f}s, Total Time: {total_training_time:.2f}s, GPU Memory: {gpu_mem_usage:.2f} MB")

    return {
        "model": model_name,
        "device": target_device.type,
        "batch_size": batch_size,
        "num_workers": num_workers,
        "total_time": total_training_time,
        "avg_epoch_time": avg_epoch_time,
        "gpu_memory_mb": gpu_mem_usage,
    }

In [ ]:
# --- Part 1 Experiment ---
print("--- Starting Part 1: CPU vs. GPU ---")
# Ensure the GPU is available for the second run
if device.type != 'cuda':
    print("WARNING: GPU not available. Skipping GPU run.")
    cpu_results = run_experiment(model_name='medium', target_device=torch.device('cpu'), batch_size=128, epochs=3, num_workers=2)
    gpu_results = cpu_results.copy() # Avoid error if no GPU
    gpu_results['device'] = 'cuda'
    gpu_results['total_time'] = float('inf') # Mark as infinitely slow
else:
    cpu_results = run_experiment(model_name='medium', target_device=torch.device('cpu'), batch_size=128, epochs=3, num_workers=2)
    gpu_results = run_experiment(model_name='medium', target_device=device, batch_size=128, epochs=3, num_workers=2)

# --- Analysis ---
cpu_time = cpu_results['total_time']
gpu_time = gpu_results['total_time']
speedup = cpu_time / gpu_time if gpu_time > 0 else float('inf')

part1_df = pd.DataFrame([cpu_results, gpu_results])
print("\n--- Part 1 Results ---")
print(part1_df[['device', 'total_time', 'avg_epoch_time', 'gpu_memory_mb']])
print(f"\nSpeedup (CPU Time / GPU Time): {speedup:.2f}x")

--- Starting Part 1: CPU vs. GPU ---


100%|██████████| 170M/170M [00:03<00:00, 49.0MB/s]


In [ ]:
# --- Part 2 Experiment ---
print("\n--- Starting Part 2: Effect of Batch Size ---")
batch_sizes = [16, 64, 256, 1024]
part2_results = []
for bs in batch_sizes:
    if device.type == 'cuda':
      res = run_experiment(model_name='medium', target_device=device, batch_size=bs, epochs=3, num_workers=2)
      part2_results.append(res)
    else:
      print("Skipping batch size experiments as no GPU is available.")
      break

part2_df = pd.DataFrame(part2_results)

# --- Plotting ---
if not part2_df.empty:
    fig, ax1 = plt.subplots(figsize=(10, 6))

    # Plot Training Time
    color = 'tab:blue'
    ax1.set_xlabel('Batch Size')
    ax1.set_ylabel('Total Training Time (s)', color=color)
    ax1.plot(part2_df['batch_size'], part2_df['total_time'], color=color, marker='o', label='Training Time')
    ax1.tick_params(axis='y', labelcolor=color)

    # Create a second y-axis for GPU Memory
    ax2 = ax1.twinx()
    color = 'tab:red'
    ax2.set_ylabel('Peak GPU Memory (MB)', color=color)
    ax2.plot(part2_df['batch_size'], part2_df['gpu_memory_mb'], color=color, marker='s', label='GPU Memory')
    ax2.tick_params(axis='y', labelcolor=color)

    plt.title('Effect of Batch Size on Training Time and GPU Memory')
    fig.tight_layout()
    plt.show()

    print("\n--- Part 2 Results ---")
    print(part2_df[['batch_size', 'total_time', 'gpu_memory_mb']])

In [ ]:
# --- Part 3 Experiment ---
print("\n--- Starting Part 3: Effect of Model Complexity ---")
model_sizes = ['small', 'medium', 'large']
part3_results = []
for model_name in model_sizes:
    if device.type == 'cuda':
      res = run_experiment(model_name=model_name, target_device=device, batch_size=128, epochs=3, num_workers=2)
      part3_results.append(res)
    else:
      print("Skipping model complexity experiments as no GPU is available.")
      break

part3_df = pd.DataFrame(part3_results)

# --- Plotting ---
if not part3_df.empty:
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))

    # Plot 1: Total Time vs. Model Size
    sns.barplot(x='model', y='total_time', data=part3_df, ax=ax[0], palette='viridis')
    ax[0].set_title('Training Time vs. Model Complexity')
    ax[0].set_ylabel('Total Training Time (s)')
    ax[0].set_xlabel('Model Size')

    # Plot 2: GPU Memory vs. Model Size
    sns.barplot(x='model', y='gpu_memory_mb', data=part3_df, ax=ax[1], palette='plasma')
    ax[1].set_title('GPU Memory Usage vs. Model Complexity')
    ax[1].set_ylabel('Peak GPU Memory (MB)')
    ax[1].set_xlabel('Model Size')

    plt.tight_layout()
    plt.show()

    print("\n--- Part 3 Results ---")
    print(part3_df[['model', 'total_time', 'gpu_memory_mb']])

In [ ]:
# --- Part 4 Experiment ---
print("\n--- Starting Part 4: Data Loading Bottlenecks ---")
worker_counts = [0, 2, 4, 8]
part4_results = []
for workers in worker_counts:
    if device.type == 'cuda':
      res = run_experiment(model_name='medium', target_device=device, batch_size=256, epochs=3, num_workers=workers)
      part4_results.append(res)
    else:
      print("Skipping data loading experiments as no GPU is available.")
      break

part4_df = pd.DataFrame(part4_results)

# --- Plotting ---
if not part4_df.empty:
    plt.figure(figsize=(8, 5))
    sns.lineplot(x='num_workers', y='total_time', data=part4_df, marker='o')
    plt.title('Effect of `num_workers` on Total Training Time')
    plt.xlabel('Number of Data Loader Workers')
    plt.ylabel('Total Training Time (s)')
    plt.xticks(worker_counts)
    plt.show()

    print("\n--- Part 4 Results ---")
    print(part4_df[['num_workers', 'total_time']])